# Transformer from scratch — exercises + automated tests
Adapted from the supplied TARA/ARENA transformer exercises. **This is the exercise version, not the completed walkthrough.** Implement the `YOUR CODE HERE` bodies. All constructors, test utilities, and test data are supplied inline. Only **PyTorch** is required in your Python 3.10+ notebook kernel; no course repositories, reference downloads, logins, or external test files.

1. Run setup and scaffolding, then work through the numbered exercises in order.
2. Replace each `raise NotImplementedError(...)` with your implementation. Keep its signature and assignment line.
3. Run your implementation cell and the following test cell. You will see **PASS**, **TODO** (unfinished), or **FAIL** (incorrect or blocked by an earlier exercise), with error details.
4. After edits, rerun the final suite; it reruns every test against current definitions rather than trusting old results. It raises an assertion until all exercises pass.

**Run All on an untouched copy intentionally does not pass.** Individual checks let you inspect the whole notebook without stopping; the final gate enforces completion. A later test can depend on an earlier exercise: finish the early ones first. Test cases are visible, but no completed solution methods are embedded. The separate completed notebook remains a reference.

The offline exercises use tiny random models and a UTF-8 byte tokenizer. They cover the core transformer, loss/training, sampling, beam search, and cache equivalence; exact pretrained GPT-2 text completions are deliberately replaced by numerical tests. Passing these tests is strong feedback, not a proof for every possible input.

In [21]:
# If needed, uncomment and run once, then restart the kernel:
# %pip install torch
import math
from dataclasses import dataclass
from pathlib import Path
import torch as t
from torch import nn
import math
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

t.manual_seed(0)
device = t.device("cpu")
print("PyTorch", t.__version__)

TESTS = {}
def register_test(name):
    def decorate(fn):
        TESTS[name] = fn
        return fn
    return decorate

def check(name):
    # Reset RNG per test so execution order does not change test data.
    t.manual_seed(1729)
    try:
        TESTS[name]()
    except NotImplementedError as exc:
        print(f"TODO {name}: {exc}")
        return False
    except Exception as exc:
        print(f"FAIL {name}: {type(exc).__name__}: {exc}")
        return False
    print(f"PASS {name}")
    return True

def run_all_tests():
    results = {name: check(name) for name in TESTS}
    passed = sum(results.values())
    print(f"\n{passed}/{len(results)} exercise test groups passed")
    assert all(results.values()), "Complete or fix the TODO/FAIL exercises above, then rerun this cell."
    return results

def tiny_cfg(**kwargs):
    defaults = dict(d_model=8, n_heads=2, d_head=4, d_mlp=16, n_layers=2, n_ctx=16, d_vocab=11)
    return Config(**(defaults | kwargs))

def close(actual, expected):
    t.testing.assert_close(actual, expected, atol=1e-6, rtol=1e-5)

PyTorch 2.14.0


## Supplied scaffolding
Run this cell once after a kernel restart. It defines parameter shapes, the byte tokenizer, and the configuration. The constructors are complete; exercise methods below are initially unfinished. **Do not rerun this scaffolding cell after solving exercises unless you then rerun your implementation cells**, because it resets the classes.

Attention returns `(output, present_kv, pattern)`. A block returns `(residual, present_kv, pattern)`. The transformer returns logits by default, or `(logits, cache_or_None, patterns_or_None)` when requested. These contracts let the tests check internal behavior without any external model.

In [2]:
@dataclass
class Config:
    d_model: int = 64
    d_vocab: int = 257
    n_ctx: int = 128
    d_head: int = 16
    n_heads: int = 4
    d_mlp: int = 256
    n_layers: int = 2
    layer_norm_eps: float = 1e-05
    init_range: float = 0.02

    def __post_init__(self):
        assert self.d_model == self.n_heads * self.d_head
        assert min(self.d_model, self.d_vocab, self.n_ctx, self.d_mlp, self.n_layers) > 0

In [3]:
class ByteTokenizer:
    vocab_size = 257
    eos_token_id = bos_token_id = 256

    def encode(self, text, return_tensors=None):
        ids = list(text.encode('utf-8'))
        return t.tensor([ids], dtype=t.long) if return_tensors == 'pt' else ids

    def decode(self, ids):
        if isinstance(ids, t.Tensor):
            ids = ids.detach().cpu().reshape(-1).tolist()
        if isinstance(ids, int):
            ids = [ids]
        return bytes((i for i in ids if i != self.eos_token_id)).decode('utf-8', errors='replace')

In [4]:
class LayerNorm(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.eps = cfg.layer_norm_eps
        self.w = nn.Parameter(t.ones(cfg.d_model))
        self.b = nn.Parameter(t.zeros(cfg.d_model))

    def forward(self, x):
        raise NotImplementedError('Implement LayerNorm.forward')

In [5]:
class Embed(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.W_E = nn.Parameter(t.randn(cfg.d_vocab, cfg.d_model) * cfg.init_range)

    def forward(self, tokens):
        raise NotImplementedError('Implement Embed.forward')

In [6]:
class PosEmbed(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.W_pos = nn.Parameter(t.randn(cfg.n_ctx, cfg.d_model) * cfg.init_range)

    def forward(self, tokens, offset=0):
        raise NotImplementedError('Implement PosEmbed.forward')

In [7]:
class Attention(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        for name in ('Q', 'K', 'V'):
            setattr(self, 'W_' + name, nn.Parameter(t.randn(cfg.n_heads, cfg.d_model, cfg.d_head) * cfg.init_range))
            setattr(self, 'b_' + name, nn.Parameter(t.zeros(cfg.n_heads, cfg.d_head)))
        self.W_O = nn.Parameter(t.randn(cfg.n_heads, cfg.d_head, cfg.d_model) * cfg.init_range)
        self.b_O = nn.Parameter(t.zeros(cfg.d_model))

    def apply_causal_mask(self, scores):
        raise NotImplementedError('Implement Attention.apply_causal_mask')

    def forward(self, x, past=None):
        raise NotImplementedError('Implement Attention.forward')

In [8]:
class MLP(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.W_in = nn.Parameter(t.randn(cfg.d_model, cfg.d_mlp) * cfg.init_range)
        self.b_in = nn.Parameter(t.zeros(cfg.d_mlp))
        self.W_out = nn.Parameter(t.randn(cfg.d_mlp, cfg.d_model) * cfg.init_range)
        self.b_out = nn.Parameter(t.zeros(cfg.d_model))

    def forward(self, x):
        raise NotImplementedError('Implement MLP.forward')

In [9]:
class TransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = (LayerNorm(cfg), LayerNorm(cfg))
        self.attn, self.mlp = (Attention(cfg), MLP(cfg))

    def forward(self, x, past=None):
        raise NotImplementedError('Implement TransformerBlock.forward')

In [10]:
class Unembed(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.W_U = nn.Parameter(t.randn(cfg.d_model, cfg.d_vocab) * cfg.init_range)
        self.b_U = nn.Parameter(t.zeros(cfg.d_vocab), requires_grad=False)

    def forward(self, x):
        raise NotImplementedError('Implement Unembed.forward')

In [11]:
class DemoTransformer(nn.Module):

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.embed, self.pos_embed = (Embed(cfg), PosEmbed(cfg))
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_final, self.unembed = (LayerNorm(cfg), Unembed(cfg))

    def forward(self, tokens, kv_cache=None, return_cache=False, return_patterns=False):
        raise NotImplementedError('Implement DemoTransformer.forward')

In [12]:
@dataclass
class TransformerTrainingArgs:
    epochs: int = 2
    max_steps_per_epoch: int = 12
    lr: float = 0.001
    weight_decay: float = 0.01

In [13]:
class TransformerTrainer:

    def __init__(self, args, model, train_loader, val_loader):
        self.args, self.model = (args, model)
        self.train_loader, self.val_loader = (train_loader, val_loader)
        self.optimizer = t.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
        self.history = []

    def training_step(self, batch):
        raise NotImplementedError('Implement TransformerTrainer.training_step')

    @t.inference_mode()
    def evaluate(self):
        raise NotImplementedError('Implement TransformerTrainer.evaluate')

    def train(self):
        for epoch in range(self.args.epochs):
            losses = []
            for step, (batch,) in enumerate(self.train_loader):
                if step >= self.args.max_steps_per_epoch:
                    break
                losses.append(self.training_step(batch))
            metrics = self.evaluate()
            row = {'epoch': epoch + 1, 'train_loss': sum(losses) / len(losses), **metrics}
            self.history.append(row)
            print(row)
        return self.history

In [14]:
class TransformerSampler:

    def __init__(self, model, tokenizer):
        self.model, self.tokenizer, self.cfg = (model, tokenizer, model.cfg)

    @staticmethod
    def greedy_search(logits):
        raise NotImplementedError('Implement TransformerSampler.greedy_search')

    @staticmethod
    def apply_temperature(logits, temperature):
        raise NotImplementedError('Implement TransformerSampler.apply_temperature')

    @staticmethod
    def apply_frequency_penalty(input_ids, logits, freq_penalty):
        raise NotImplementedError('Implement TransformerSampler.apply_frequency_penalty')

    @staticmethod
    def sample_basic(logits):
        raise NotImplementedError('Implement TransformerSampler.sample_basic')

    @staticmethod
    def sample_top_k(logits, k):
        raise NotImplementedError('Implement TransformerSampler.sample_top_k')

    @staticmethod
    def sample_top_p(logits, top_p, min_tokens_to_keep=1):
        raise NotImplementedError('Implement TransformerSampler.sample_top_p')

    @staticmethod
    def sample_next_token(input_ids, logits, temperature=1.0, top_k=0, top_p=0.0, frequency_penalty=0.0):
        raise NotImplementedError('Implement TransformerSampler.sample_next_token')

    @t.inference_mode()
    def sample(self, prompt, max_tokens_generated=40, seed=None, use_cache=False, **kwargs):
        raise NotImplementedError('Implement TransformerSampler.sample')

In [16]:
tokenizer = ByteTokenizer()
cfg = tiny_cfg()

## Exercise 1: Layer normalization

Normalize the final dimension using population variance, epsilon inside the square root, and learned scale `w` and bias `b`. Input/output: `(batch, position, d_model)`. Do not detach gradients.

Edit the next cell, then run its test cell.

In [26]:
def LayerNorm_forward(self, x):
    mean = x.mean(dim = -1, keepdim=True)
    variance = pow((x - mean),2).mean(dim = -1, keepdim = True)
    norm = (self.w * (x - mean) / t.sqrt(variance + self.eps)) + self.b
    return norm

LayerNorm.forward = LayerNorm_forward

In [27]:
@register_test('LayerNorm.forward')
def test_1():
    c = tiny_cfg(layer_norm_eps=1e-3)
    m = LayerNorm(c).double()
    with t.no_grad():
        m.w.copy_(t.randn_like(m.w)); m.b.copy_(t.randn_like(m.b))
    for x in [t.randn(2,3,c.d_model,dtype=t.float64), t.ones(1,2,c.d_model,dtype=t.float64), t.randn(2,1,c.d_model,dtype=t.float64)*1e-4]:
        x.requires_grad_()
        actual = m(x)
        expected = F.layer_norm(x, (c.d_model,), m.w, m.b, c.layer_norm_eps)
        close(actual,expected)
        weights=t.randn_like(actual)
        ga=t.autograd.grad((actual*weights).sum(),x,retain_graph=True)[0]
        ge=t.autograd.grad((expected*weights).sum(),x)[0]
        close(ga,ge)

check('LayerNorm.forward')

PASS LayerNorm.forward


True

## Exercise 2: Token embeddings

Index `W_E` with integer token IDs. Input `(batch, position)` → output `(batch, position, d_model)`. Repeated IDs must accumulate gradients.

Edit the next cell, then run its test cell.

In [ ]:
def Embed_forward(self, tokens):
    # YOUR CODE HERE
    raise NotImplementedError('Embed.forward: YOUR CODE HERE')

Embed.forward = Embed_forward

In [ ]:
@register_test('Embed.forward')
def test_2():
    m=Embed(tiny_cfg()); ids=t.tensor([[0,2,2],[10,1,0]])
    close(m(ids),F.embedding(ids,m.W_E))
    m(ids).sum().backward()
    close(m.W_E.grad[2],t.full((8,),2.0)); close(m.W_E.grad[3],t.zeros(8))

check('Embed.forward')

## Exercise 3: Positional embeddings

Return positional embeddings expanded across the batch. Start at `offset`, which is zero without a KV cache. Raise `ValueError` for negative offsets or context overflow.

Edit the next cell, then run its test cell.

In [ ]:
def PosEmbed_forward(self, tokens, offset=0):
    # YOUR CODE HERE
    raise NotImplementedError('PosEmbed.forward: YOUR CODE HERE')

PosEmbed.forward = PosEmbed_forward

In [ ]:
@register_test('PosEmbed.forward')
def test_3():
    c=tiny_cfg(); m=PosEmbed(c); ids=t.zeros(3,4,dtype=t.long)
    for off in (0,3,12):
        close(m(ids,off),m.W_pos[off:off+4][None].expand(3,-1,-1))
    for off in (-1,13):
        try: m(ids,off)
        except ValueError: pass
        else: raise AssertionError("Invalid offset must raise ValueError")

check('PosEmbed.forward')

## Exercise 4: Causal mask

Scores have shape `(batch, head, query, key)`. Mask future keys with negative infinity without modifying the input. For cached attention, query positions start at `key_length - query_length`.

Edit the next cell, then run its test cell.

In [34]:
def Attention_apply_causal_mask(self, scores):
    print(scores.shape[-2:])
    nq, nk = scores.shape[-2:]
    mask = t.ones((nq, nk), dtype = t.bool, device = scores.device)
    mask = mask.triu(diagonal = nk - nq + 1)
    return scores.masked_fill(mask, float("-inf"))
    
Attention.apply_causal_mask = Attention_apply_causal_mask



In [35]:
@register_test('Attention.apply_causal_mask')
def test_4():
    m=Attention(tiny_cfg())
    for nq,nk in [(4,4),(2,6),(1,6)]:
        scores=t.randn(2,2,nq,nk); original=scores.clone()
        actual=m.apply_causal_mask(scores)
        close(scores,original)
        for q in range(nq):
            for k in range(nk):
                if k>nk-nq+q: assert t.isneginf(actual[:,:,q,k]).all(), "Future key not masked"
                else: close(actual[:,:,q,k],scores[:,:,q,k])

check('Attention.apply_causal_mask')

torch.Size([4, 4])
torch.Size([2, 6])
torch.Size([1, 6])
PASS Attention.apply_causal_mask


True

## Exercise 5: Multi-head attention and KV cache

Project Q/K/V with the supplied weights, scale dot products by `sqrt(d_head)`, apply the causal mask, softmax, combine values, and project through `W_O` plus `b_O`. Return `(output, (k,v), pattern)`. K/V shape: `(batch, heads, total_positions, d_head)`. If `past` is supplied, concatenate old and new K/V on the position axis. Input contains only new positions.

Edit the next cell, then run its test cell.

In [ ]:
def Attention_forward(self, x, past=None):
    

Attention.forward = Attention_forward

In [ ]:
@register_test('Attention.forward')
def test_5():
    c=tiny_cfg(); m=Attention(c); x=t.randn(2,5,c.d_model)
    with t.no_grad():
        for name in ['b_Q','b_K','b_V','b_O']: getattr(m,name).normal_()
    out,present,pattern=m(x)
    qkv=[t.einsum('bpm,hmd->bhpd',x,getattr(m,'W_'+n))+getattr(m,'b_'+n)[None,:,None,:] for n in 'QKV']
    z=F.scaled_dot_product_attention(*qkv,is_causal=True)
    expected=t.einsum('bhpd,hdm->bpm',z,m.W_O)+m.b_O
    close(out,expected); close(present[0],qkv[1]); close(present[1],qkv[2])
    assert pattern.shape==(2,2,5,5)
    close(pattern.sum(-1),t.ones(2,2,5)); assert t.count_nonzero(pattern.triu(1))==0
    first,cache,_=m(x[:,:3]); rest,_,_=m(x[:,3:],cache)
    close(t.cat([first,rest],1),out)

check('Attention.forward')

## Exercise 6: MLP

Apply the input affine projection, GPT-2 tanh-approximate GELU, and output affine projection. Input/output shape is unchanged.

Edit the next cell, then run its test cell.

In [ ]:
def MLP_forward(self, x):
    pass

MLP.forward = MLP_forward

In [ ]:
@register_test('MLP.forward')
def test_6():
    m=MLP(tiny_cfg()); x=t.randn(2,3,8)
    with t.no_grad(): m.b_in.normal_(); m.b_out.normal_()
    expected=F.linear(F.gelu(F.linear(x,m.W_in.T,m.b_in),approximate='tanh'),m.W_out.T,m.b_out)
    close(m(x),expected)

check('MLP.forward')

## Exercise 7: Pre-norm residual block

Apply attention to `ln1(x)` and add it to x. Apply the MLP to `ln2` of that intermediate residual and add again. Return the residual, updated KV cache, and attention pattern.

Edit the next cell, then run its test cell.

In [ ]:
def TransformerBlock_forward(self, x, past=None):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerBlock.forward: YOUR CODE HERE')

TransformerBlock.forward = TransformerBlock_forward

In [ ]:
@register_test('TransformerBlock.forward')
def test_7():
    c=tiny_cfg(); m=TransformerBlock(c); x=t.randn(2,5,8)
    a,cache,pattern=m.attn(m.ln1(x)); mid=x+a
    actual,present,p=m(x)
    close(actual,mid+m.mlp(m.ln2(mid))); close(p,pattern)
    close(present[0],cache[0]); close(present[1],cache[1])
    f,past,_=m(x[:,:2]); r,_,_=m(x[:,2:],past)
    close(t.cat([f,r],1),actual)

check('TransformerBlock.forward')

## Exercise 8: Unembedding

Project the normalized residual stream into vocabulary logits using `W_U` and `b_U`. Do not apply softmax.

Edit the next cell, then run its test cell.

In [ ]:
def Unembed_forward(self, x):
    # YOUR CODE HERE
    raise NotImplementedError('Unembed.forward: YOUR CODE HERE')

Unembed.forward = Unembed_forward

In [ ]:
@register_test('Unembed.forward')
def test_8():
    m=Unembed(tiny_cfg()); x=t.randn(2,4,8)
    with t.no_grad(): m.b_U.normal_()
    close(m(x),F.linear(x,m.W_U.T,m.b_U))
    assert m(x).shape==(2,4,11)

check('Unembed.forward')

## Exercise 9: Full transformer, with cache support

Sum token and positional embeddings, run all blocks, then final normalization and unembedding. Default return is logits. If either return flag is True, return `(logits, cache if requested else None, patterns if requested else None)`. Cached input contains only new tokens; offset positions by cached length. Reject empty sequences, incorrect cache length/shape, and context overflow with `ValueError`.

Edit the next cell, then run its test cell.

In [ ]:
def DemoTransformer_forward(self, tokens, kv_cache=None, return_cache=False, return_patterns=False):
    # YOUR CODE HERE
    raise NotImplementedError('DemoTransformer.forward: YOUR CODE HERE')

DemoTransformer.forward = DemoTransformer_forward

In [ ]:
@register_test('DemoTransformer.forward')
def test_9():
    c=tiny_cfg(); m=DemoTransformer(c); ids=t.randint(0,c.d_vocab,(2,7))
    x=m.embed(ids)+m.pos_embed(ids)
    for block in m.blocks: x,_,_=block(x)
    close(m(ids),m.unembed(m.ln_final(x)))
    full,cache,patterns=m(ids,return_cache=True,return_patterns=True)
    assert len(cache)==c.n_layers and len(patterns)==c.n_layers
    changed=ids.clone(); changed[:,4:]=(changed[:,4:]+1)%c.d_vocab
    close(full[:,:4],m(changed)[:,:4])
    f,past,_=m(ids[:,:3],return_cache=True)
    r,_,_=m(ids[:,3:],kv_cache=past,return_cache=True)
    close(t.cat([f,r],1),full)
    cache=None; parts=[]
    for i in range(ids.shape[1]):
        out,cache,_=m(ids[:,i:i+1],kv_cache=cache,return_cache=True); parts.append(out)
    close(t.cat(parts,1),full)
    for bad in [t.empty(1,0,dtype=t.long),t.zeros(1,17,dtype=t.long)]:
        try: m(bad)
        except ValueError: pass
        else: raise AssertionError("Invalid sequence must raise ValueError")
    try: m(ids[:,:1],kv_cache=[])
    except ValueError: pass
    else: raise AssertionError("Invalid cache must raise ValueError")

check('DemoTransformer.forward')

## Exercise 10: Shifted next-token log probabilities

Return shape `(batch, sequence-1)`: position i logits score token i+1. Use log probabilities rather than probabilities.

Edit the next cell, then run its test cell.

In [ ]:
def get_log_probs(logits, tokens):
    # YOUR CODE HERE
    raise NotImplementedError('get_log_probs: YOUR CODE HERE')

In [ ]:
@register_test('get_log_probs')
def test_10():
    logits=t.randn(2,5,11,requires_grad=True); ids=t.randint(0,11,(2,5))
    actual=get_log_probs(logits,ids)
    expected=-F.cross_entropy(logits[:,:-1].reshape(-1,11),ids[:,1:].reshape(-1),reduction='none').reshape(2,4)
    close(actual,expected)
    actual.sum().backward(); assert t.isfinite(logits.grad).all()
    assert get_log_probs(logits[:,:1],ids[:,:1]).shape==(2,0)

check('get_log_probs')

## Exercise 11: Training step

The batch is a tensor `(batch, sequence+1)`. Move it to the model device, enter train mode, clear gradients, compute shifted cross entropy, backpropagate, clip gradient norm to 1.0, and step the supplied optimizer. Return the loss as a Python float.

Edit the next cell, then run its test cell.

In [ ]:
def TransformerTrainer_training_step(self, batch):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerTrainer.training_step: YOUR CODE HERE')

TransformerTrainer.training_step = TransformerTrainer_training_step

In [ ]:
@register_test('TransformerTrainer.training_step')
def test_11():
    import copy
    c=tiny_cfg(); m=DemoTransformer(c); other=copy.deepcopy(m)
    args=TransformerTrainingArgs(); trainer=TransformerTrainer(args,m,[],[])
    ids=t.randint(0,c.d_vocab,(2,8))
    optimizer=t.optim.AdamW(other.parameters(),lr=args.lr,weight_decay=args.weight_decay)
    # Compare TWO updates: forgetting zero_grad often passes a one-step test.
    for _ in range(2):
        optimizer.zero_grad(set_to_none=True)
        logits=other(ids[:,:-1]); expected=F.cross_entropy(logits.reshape(-1,c.d_vocab),ids[:,1:].reshape(-1))
        expected.backward(); nn.utils.clip_grad_norm_(other.parameters(),1.0); optimizer.step()
        value=trainer.training_step(ids)
        assert isinstance(value,float); assert abs(value-expected.item())<1e-5
        for a,b in zip(m.parameters(),other.parameters()): close(a,b)
    assert m.training

check('TransformerTrainer.training_step')

## Exercise 12: Evaluation

Use the validation loader of one-tensor tuples. Return `{"loss": mean token cross entropy, "accuracy": token accuracy}`. Weight metrics by token count, disable gradients, and restore the model's previous train/eval mode.

Edit the next cell, then run its test cell.

In [ ]:
@t.inference_mode()
def TransformerTrainer_evaluate(self):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerTrainer.evaluate: YOUR CODE HERE')

TransformerTrainer.evaluate = TransformerTrainer_evaluate

In [ ]:
@register_test('TransformerTrainer.evaluate')
def test_12():
    c=tiny_cfg(); m=DemoTransformer(c); data=t.randint(0,c.d_vocab,(5,8))
    loader=DataLoader(TensorDataset(data),batch_size=2)
    trainer=TransformerTrainer(TransformerTrainingArgs(),m,[],loader)
    with t.no_grad():
        logits=m(data[:,:-1]); targets=data[:,1:]
        expected=F.cross_entropy(logits.reshape(-1,c.d_vocab),targets.reshape(-1)).item()
        accuracy=(logits.argmax(-1)==targets).float().mean().item()
    for mode in (True,False):
        m.train(mode); result=trainer.evaluate()
        assert abs(result['loss']-expected)<1e-5 and abs(result['accuracy']-accuracy)<1e-6
        assert m.training==mode, "Restore previous training mode"
    assert all(p.grad is None for p in m.parameters())

check('TransformerTrainer.evaluate')

## Exercise 13: Greedy sampling

Return the index of the largest logit as a Python int. For ties, use the first maximum.

Edit the next cell, then run its test cell.

In [ ]:
def TransformerSampler_greedy_search(logits):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerSampler.greedy_search: YOUR CODE HERE')

TransformerSampler.greedy_search = staticmethod(TransformerSampler_greedy_search)

In [ ]:
@register_test('TransformerSampler.greedy_search')
def test_13():
    assert TransformerSampler.greedy_search(t.tensor([-5.,3.,2.]))==1
    assert TransformerSampler.greedy_search(t.tensor([2.,2.,1.]))==0
    assert isinstance(TransformerSampler.greedy_search(t.zeros(3)),int)

check('TransformerSampler.greedy_search')

## Exercise 14: Temperature

Divide logits by a strictly positive temperature without modifying the input. Raise `ValueError` for zero or negative temperature; greedy decoding is handled by the dispatcher.

Edit the next cell, then run its test cell.

In [ ]:
def TransformerSampler_apply_temperature(logits, temperature):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerSampler.apply_temperature: YOUR CODE HERE')

TransformerSampler.apply_temperature = staticmethod(TransformerSampler_apply_temperature)

In [ ]:
@register_test('TransformerSampler.apply_temperature')
def test_14():
    x=t.tensor([-2.,0.,3.]); original=x.clone()
    for temp in (0.01,1.,10.): close(TransformerSampler.apply_temperature(x,temp),x/temp)
    close(x,original)
    for temp in (0.,-1.):
        try: TransformerSampler.apply_temperature(x,temp)
        except ValueError: pass
        else: raise AssertionError("Nonpositive temperature must fail")

check('TransformerSampler.apply_temperature')

## Exercise 15: Frequency penalty

Subtract `freq_penalty × token_count` from each vocabulary logit. Count all input IDs, including repeats. Preserve the input logits.

Edit the next cell, then run its test cell.

In [ ]:
def TransformerSampler_apply_frequency_penalty(input_ids, logits, freq_penalty):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerSampler.apply_frequency_penalty: YOUR CODE HERE')

TransformerSampler.apply_frequency_penalty = staticmethod(TransformerSampler_apply_frequency_penalty)

In [ ]:
@register_test('TransformerSampler.apply_frequency_penalty')
def test_15():
    ids=t.tensor([0,2,2,2]); x=t.tensor([1.,2.,3.,4.]); original=x.clone()
    close(TransformerSampler.apply_frequency_penalty(ids,x,0.5),t.tensor([0.5,2.,1.5,4.]))
    close(TransformerSampler.apply_frequency_penalty(ids,x,-1),t.tensor([2.,2.,6.,4.]))
    close(x,original)
    close(TransformerSampler.apply_frequency_penalty(t.empty(0,dtype=t.long),x,3),x)

check('TransformerSampler.apply_frequency_penalty')

## Exercise 16: Categorical sampling

Draw one token from softmax(logits) and return its Python integer index. The frequency test uses a fixed seed and a generous tolerance.

Edit the next cell, then run its test cell.

In [ ]:
def TransformerSampler_sample_basic(logits):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerSampler.sample_basic: YOUR CODE HERE')

TransformerSampler.sample_basic = staticmethod(TransformerSampler_sample_basic)

In [ ]:
@register_test('TransformerSampler.sample_basic')
def test_16():
    logits=t.tensor([0.1,0.3,0.6]).log()
    samples=[TransformerSampler.sample_basic(logits) for _ in range(4000)]
    assert all(isinstance(i,int) and 0<=i<3 for i in samples)
    freq=t.bincount(t.tensor(samples),minlength=3)/len(samples)
    t.testing.assert_close(freq,logits.exp(),atol=0.035,rtol=0)

check('TransformerSampler.sample_basic')

## Exercise 17: Top-k sampling

Sample only among the k greatest logits, with probabilities renormalized over those entries. Return original vocabulary indices. Reject k outside `[1, vocabulary size]`.

Edit the next cell, then run its test cell.

In [ ]:
def TransformerSampler_sample_top_k(logits, k):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerSampler.sample_top_k: YOUR CODE HERE')

TransformerSampler.sample_top_k = staticmethod(TransformerSampler_sample_top_k)

In [ ]:
@register_test('TransformerSampler.sample_top_k')
def test_17():
    x=t.tensor([0.05,0.6,0.1,0.25]).log()
    assert all(TransformerSampler.sample_top_k(x,1)==1 for _ in range(20))
    samples=t.tensor([TransformerSampler.sample_top_k(x,2) for _ in range(3000)])
    assert set(samples.tolist())=={1,3}
    assert abs((samples==1).float().mean().item()-0.6/0.85)<0.04
    for k in (0,5):
        try: TransformerSampler.sample_top_k(x,k)
        except ValueError: pass
        else: raise AssertionError("Invalid k must fail")

check('TransformerSampler.sample_top_k')

## Exercise 18: Nucleus sampling

Keep the smallest descending-probability prefix whose cumulative probability reaches top_p, including the crossing token, and at least `min_tokens_to_keep`. Sample from the renormalized kept logits and return original indices. Require `0 < top_p <= 1` and a valid minimum count.

Edit the next cell, then run its test cell.

In [ ]:
def TransformerSampler_sample_top_p(logits, top_p, min_tokens_to_keep=1):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerSampler.sample_top_p: YOUR CODE HERE')

TransformerSampler.sample_top_p = staticmethod(TransformerSampler_sample_top_p)

In [ ]:
@register_test('TransformerSampler.sample_top_p')
def test_18():
    x=t.tensor([0.1,0.6,0.3]).log()
    assert all(TransformerSampler.sample_top_p(x,0.2)==1 for _ in range(30))
    samples=t.tensor([TransformerSampler.sample_top_p(x,0.7) for _ in range(3000)])
    assert set(samples.tolist())=={1,2}, "Keep crossing token; exclude tail"
    assert abs((samples==1).float().mean().item()-2/3)<0.04
    assert set(TransformerSampler.sample_top_p(x,0.1,2) for _ in range(200))=={1,2}
    assert set(TransformerSampler.sample_top_p(x,1.0) for _ in range(300))=={0,1,2}

check('TransformerSampler.sample_top_p')

## Exercise 19: Sampling dispatcher

Validate temperature, top-k, and top-p. Top-k and top-p cannot both be enabled. Apply frequency penalty first; if temperature is zero, return greedy output. Otherwise scale temperature, then select top-k, top-p, or basic sampling. For this notebook, frequency penalty also affects greedy decoding.

Edit the next cell, then run its test cell.

In [ ]:
def TransformerSampler_sample_next_token(input_ids, logits, temperature=1.0, top_k=0, top_p=0.0, frequency_penalty=0.0):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerSampler.sample_next_token: YOUR CODE HERE')

TransformerSampler.sample_next_token = staticmethod(TransformerSampler_sample_next_token)

In [ ]:
@register_test('TransformerSampler.sample_next_token')
def test_19():
    S=TransformerSampler; ids=t.tensor([2,2]); x=t.tensor([1.,2.,3.])
    assert S.sample_next_token(ids,x,temperature=0)==2
    assert S.sample_next_token(ids,x,temperature=0,frequency_penalty=2)==1
    assert all(S.sample_next_token(ids,x,top_k=1)==2 for _ in range(20))
    for kwargs in [dict(temperature=-1),dict(top_k=4),dict(top_p=1.1),dict(top_k=1,top_p=0.8)]:
        try: S.sample_next_token(ids,x,**kwargs)
        except ValueError: pass
        else: raise AssertionError(f"Must reject {kwargs}")
    # Check dispatch equals direct calls with the same RNG state.
    for kwargs in [{},{'top_k':2},{'top_p':0.8}]:
        t.manual_seed(5); got=S.sample_next_token(ids,x,temperature=2,frequency_penalty=0.2,**kwargs)
        adjusted=S.apply_temperature(S.apply_frequency_penalty(ids,x,0.2),2)
        t.manual_seed(5)
        expected=S.sample_top_k(adjusted,2) if 'top_k' in kwargs else S.sample_top_p(adjusted,0.8) if 'top_p' in kwargs else S.sample_basic(adjusted)
        assert got==expected

check('TransformerSampler.sample_next_token')

## Exercise 20: Autoregressive generation

Encode the prompt (use BOS for an empty prompt), repeatedly predict the final-position next token, append it, and stop on EOS or the token limit. Decode the complete sequence. Respect `seed`, pass sampling kwargs through, use eval/inference mode, and crop to context length. With `use_cache=True`, pass only new tokens after the initial prefix; rebuild the cache from the cropped context when full.

Edit the next cell, then run its test cell.

In [ ]:
@t.inference_mode()
def TransformerSampler_sample(self, prompt, max_tokens_generated=40, seed=None, use_cache=False, **kwargs):
    # YOUR CODE HERE
    raise NotImplementedError('TransformerSampler.sample: YOUR CODE HERE')

TransformerSampler.sample = TransformerSampler_sample

In [ ]:
@register_test('TransformerSampler.sample')
def test_20():
    c=tiny_cfg(d_vocab=257); m=DemoTransformer(c); s=TransformerSampler(m,ByteTokenizer())
    for prompt in ['Hi','x'*20,'']:
        a=s.sample(prompt,8,temperature=0)
        b=s.sample(prompt,8,temperature=0,use_cache=True)
        assert a==b, "Cached generation disagrees with uncached"
    assert s.sample('Hi',0)=='Hi'
    assert s.sample('Hi',8,seed=22)==s.sample('Hi',8,seed=22)
    # Deterministic logits verify appending, truncation, and EOS handling.
    class Fixed(nn.Module):
        def __init__(self,target):
            super().__init__(); self.anchor=nn.Parameter(t.zeros(1)); self.cfg=tiny_cfg(d_vocab=257,n_ctx=4); self.target=target; self.calls=0
        def forward(self,ids):
            assert ids.shape[1]<=4; self.calls+=1
            out=t.full((*ids.shape,257),-100.); out[:,:,self.target]=100.; return out
    fixed=Fixed(65); fs=TransformerSampler(fixed,ByteTokenizer())
    assert fs.sample('hello',3,temperature=0)=='helloAAA' and fixed.calls==3
    fixed=Fixed(256); fs=TransformerSampler(fixed,ByteTokenizer())
    assert fs.sample('hello',3,temperature=0)=='hello' and fixed.calls==1

check('TransformerSampler.sample')

## Exercise 21: N-gram repetition blocking

Given a Python list of token IDs and n, return a set of next-token IDs that would repeat an existing n-gram. Handle n=1, short prefixes, and disabled blocking (`None` or 0).

Edit the next cell, then run its test cell.

In [ ]:
def blocked_tokens(ids, n):
    # YOUR CODE HERE
    raise NotImplementedError('blocked_tokens: YOUR CODE HERE')

In [ ]:
@register_test('blocked_tokens')
def test_21():
    assert blocked_tokens([1,2,1],2)=={2}
    assert blocked_tokens([1,2,1],1)=={1,2}
    assert blocked_tokens([1,2],3)==set()
    assert blocked_tokens([1,2,3,1,2],3)=={3}
    assert blocked_tokens([],1)==set()
    assert blocked_tokens([1,2],None)==set()

check('blocked_tokens')

## Exercise 22: Beam search

Keep up to `num_beams` candidates by cumulative log probability (no length normalization). Completed EOS beams stay in the candidate pool. Expand active beams, apply optional n-gram blocking without renormalizing log probabilities, and retain the best candidates each round. Stop at the token limit or when all retained beams finish. Return up to `num_return_sequences` sorted `(score, decoded_text)` pairs. If all tokens are blocked, treat the beam as finished. Validate beam counts, nonnegative length, and positive n-gram size.

Edit the next cell, then run its test cell.

In [ ]:
@t.inference_mode()
def beam_search(self, prompt, num_return_sequences=2, num_beams=4, max_new_tokens=16, no_repeat_ngram_size=None):
    # YOUR CODE HERE
    raise NotImplementedError('beam_search: YOUR CODE HERE')

TransformerSampler.beam_search = beam_search

In [ ]:
@register_test('beam_search')
def test_22():
    # A tiny deterministic model makes the expected paths hand-checkable.
    class Tok:
        bos_token_id=0; eos_token_id=2
        def encode(self,text): return [0]
        def decode(self,ids): return ''.join(str(i) for i in ids if i!=2)
    class Fixed(nn.Module):
        def __init__(self):
            super().__init__(); self.anchor=nn.Parameter(t.zeros(1)); self.cfg=tiny_cfg(d_vocab=3)
        def forward(self,ids):
            return t.tensor([0.5,0.3,0.2]).log().expand(*ids.shape,3)
    s=TransformerSampler(Fixed(),Tok())
    r=s.beam_search('x',num_return_sequences=2,num_beams=3,max_new_tokens=2)
    assert [text for _,text in r]==['000','0'], r
    assert abs(r[0][0]-math.log(0.25))<1e-6 and abs(r[1][0]-math.log(0.2))<1e-6
    r=s.beam_search('x',num_return_sequences=1,num_beams=1,max_new_tokens=3,no_repeat_ngram_size=1)
    assert r[0][1]=='01', r
    assert s.beam_search('x',1,1,0)==[(0.0,'0')]
    for kwargs in [dict(num_return_sequences=3,num_beams=2),dict(max_new_tokens=-1),dict(no_repeat_ngram_size=0)]:
        try: s.beam_search('x',**kwargs)
        except ValueError: pass
        else: raise AssertionError("Invalid beam arguments must fail")

check('beam_search')

## Final completion gate
Rerun this cell whenever you change an answer. It reruns all registered tests. If you restarted the kernel, first run the setup, scaffolding, implementation, and test-definition cells above. A complete solution should pass every group. These tests include reference comparisons, gradients, repeated optimizer updates, causal masking, cached chunk equivalence, sampling distributions, and deterministic beam outcomes.

In [ ]:
assert len(TESTS) == 22, "Run all preceding test-definition cells first."
results = run_all_tests()

## Optional end-to-end training demo (after all tests pass)
This exercises your implementations on a small built-in corpus with no downloads. Expect only rudimentary text from such a tiny run. Training configuration and data loading are provided; the training/evaluation methods are yours.

In [ ]:
text = ("Once upon a time a robot read a book. The robot shared a story with a friend. ") * 30
ids = t.tensor(tokenizer.encode(text), dtype=t.long)
windows = ids.unfold(0, 33, 32).contiguous()
# Tiny smoke demo only; held-out windows come from repeated text, not a generalization benchmark.
split = int(len(windows)*0.8)
train_loader = DataLoader(TensorDataset(windows[:split]),batch_size=8,shuffle=True)
val_loader = DataLoader(TensorDataset(windows[split:]),batch_size=8)
model = DemoTransformer(Config(d_model=32,n_heads=2,d_head=16,d_mlp=64,n_layers=2,n_ctx=64))
trainer = TransformerTrainer(TransformerTrainingArgs(epochs=2,max_steps_per_epoch=8),model,train_loader,val_loader)
print("Before:",trainer.evaluate())
history = trainer.train()
print(TransformerSampler(model,tokenizer).sample("Once upon a time",max_tokens_generated=24,temperature=0.7,top_k=10,seed=0))